# Replikasi Eksperimen Giudici et al. (2020)
## *Network Models to Improve Automated Cryptocurrency Portfolio Management*

Notebook ini mengimplementasikan dan membandingkan strategi portofolio berikut:
1. **Equally Weighted (EW)** — Portofolio naif 1/N
2. **Classical Markowitz (CM)** — Optimasi Mean-Variance tradisional
3. **Glasso Markowitz (GM)** — Markowitz + regularisasi Graphical Lasso
4. **Network Markowitz (NW)** — RMT + MST + penalti sentralitas eigenvector
5. **Graph Diversification** — Seleksi aset via Maximum Independent Set (MIS)
6. **AGGP** — Adaptive Graph-Gated Portfolio (kontribusi orisinal)

**Data:** 10 aset kripto utama, periode 14 September 2017 – 17 Oktober 2019

---
## Sel 1 — Persiapan Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy.optimize import minimize
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.linalg import eigh
from sklearn.covariance import GraphicalLassoCV
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

---
## Sel 2 — Memuat Data

Memuat data return dan harga dari file Excel. Fokus utama adalah dataset return harian untuk 10 aset kripto utama dalam periode 14 September 2017 hingga 17 Oktober 2019.

In [ ]:
# Load data
excel_file = 'crypto_data_real.xlsx'
df_returns = pd.read_excel(excel_file, sheet_name='Returns', index_col=0)
df_prices  = pd.read_excel(excel_file, sheet_name='Prices',  index_col=0)

crypto_names = df_returns.columns.tolist()
n_assets     = len(crypto_names)

print(f"Data loaded: {df_returns.shape[0]} days, {n_assets} assets")
print(f"Period: {df_returns.index[0]} to {df_returns.index[-1]}")
print(f"Assets: {crypto_names}")

---
## Sel 3 — Statistik Ringkasan (Summary Statistics)

Menghasilkan tabel statistik deskriptif untuk setiap aset kripto, mencakup Mean, Standar Deviasi, Kurtosis, dan Skewness (replikasi Tabel 1 pada paper acuan).

In [ ]:
# Create summary statistics table
summary_stats = pd.DataFrame({
    'Mean':     df_returns.mean(),
    'Std':      df_returns.std(),
    'Kurtosis': df_returns.kurt(),
    'Skewness': df_returns.skew()
})

print("TABLE 1 | Summary statistics.")
print(summary_stats.round(4))

---
## Sel 4 — Visualisasi Harga Ternormalisasi (Figure 1 & 2)

Mereplikasi *"Normalized cryptocurrency price series"* dengan mengatur harga awal setiap aset menjadi 100 pada tanggal 7 Januari 2018.

In [ ]:
# --- Figure 1: BTC, ETH, USDT, BCH, LTC ---
assets_1  = ['BTC', 'ETH', 'USDT', 'BCH', 'LTC']
df_norm_1 = (df_prices.loc['2018-01-07':, assets_1] / df_prices.loc['2018-01-07', assets_1]) * 100
colors_1  = ['black', 'red', 'green', 'blue', 'cyan']

plt.figure(figsize=(12, 5))
for i, col in enumerate(df_norm_1.columns):
    plt.plot(df_norm_1.index, df_norm_1[col], label=col, color=colors_1[i])
plt.title('Figure 1 | Normalized Price Series I (BTC, ETH, USDT, BCH, LTC)')
plt.ylabel('Normalized Price (base=100)')
plt.legend()
plt.tight_layout()
plt.show()

# --- Figure 2: XRP, BNB, EOS, XLM, TRX ---
assets_2  = ['XRP', 'BNB', 'EOS', 'XLM', 'TRX']
df_norm_2 = (df_prices.loc['2018-01-07':, assets_2] / df_prices.loc['2018-01-07', assets_2]) * 100
colors_2  = ['magenta', 'gold', 'lightgrey', 'black', 'red']

plt.figure(figsize=(12, 5))
for i, col in enumerate(df_norm_2.columns):
    plt.plot(df_norm_2.index, df_norm_2[col], label=col, color=colors_2[i])
plt.title('Figure 2 | Normalized Price Series II (XRP, BNB, EOS, XLM, TRX)')
plt.ylabel('Normalized Price (base=100)')
plt.legend()
plt.tight_layout()
plt.show()

---
## Sel 5 — Visualisasi Minimum Spanning Tree (Figure 3 & 4)

Mereplikasi Figure 3 dari paper acuan: struktur jaringan aset kripto selama periode gelembung spekulatif (Sep 2017 – Jan 2018) dan periode stabil (Jun 2019 – Okt 2019). MST membantu mengidentifikasi aset yang menjadi pusat (*hub*) dalam sistem.

In [ ]:
# --- Figure 3 | MST Speculative Bubble Period ---
df_bubble    = df_returns.loc['2017-09-14':'2018-01-31']
corr_bubble  = df_bubble.corr()
dist_bubble  = np.sqrt(2 * (1 - corr_bubble))

G          = nx.from_pandas_adjacency(dist_bubble)
mst_bubble = nx.minimum_spanning_tree(G, weight='weight')

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(mst_bubble, seed=42)
nx.draw(mst_bubble, pos, with_labels=True,
        node_color='orange', node_size=1500,
        edge_color='black', linewidths=1.5,
        font_size=10, font_weight='bold')
plt.title('Figure 3 | MST September 2017 - January 2018', fontsize=12)
plt.show()

# --- Figure 4 | MST Stable Period ---
df_stable    = df_returns.loc['2019-06-01':'2019-10-17']
corr_stable  = df_stable.corr()
dist_stable  = np.sqrt(2 * (1 - corr_stable))

G2         = nx.from_pandas_adjacency(dist_stable)
mst_stable = nx.minimum_spanning_tree(G2, weight='weight')

plt.figure(figsize=(10, 8))
pos2 = nx.spring_layout(mst_stable, seed=42)
nx.draw(mst_stable, pos2, with_labels=True,
        node_color='orange', node_size=1500,
        edge_color='black', linewidths=1.5,
        font_size=10, font_weight='bold')
plt.title('Figure 4 | MST June 2019 - October 2019', fontsize=12)
plt.show()

---
## Sel 6 — Fungsi Pembantu (Helper Functions)

Implementasi:
- **RMT Filter** — penyaringan noise matriks korelasi via *Random Matrix Theory* (Marcenko-Pastur bound)
- **MST Builder** — konstruksi jarak antar aset
- **Eigenvector Centrality** — mengukur kepusatan node dalam jaringan
- **VaR, Rachev Ratio, Max Drawdown** — metrik risiko
- **Graph Diversification (MIS & RMT-MIS)** — seleksi aset berbasis topologi graf

In [ ]:
def apply_rmt_filter(returns_data):
    """Filter noise dari matriks korelasi menggunakan Random Matrix Theory."""
    data = returns_data.values if isinstance(returns_data, pd.DataFrame) else returns_data
    T, N = data.shape
    Q    = T / N
    C    = np.corrcoef(data.T)

    eigenvalues, eigenvectors = eigh(C)
    eigenvalues  = eigenvalues[::-1]
    eigenvectors = eigenvectors[:, ::-1]

    lambda_plus      = 1 + (1/Q) + 2*np.sqrt(1/Q)   # Marcenko-Pastur upper bound
    significant_mask = eigenvalues > lambda_plus
    Lambda_filtered  = np.diag(np.where(significant_mask, eigenvalues, 0))
    C_filtered       = eigenvectors @ Lambda_filtered @ eigenvectors.T
    return C_filtered


def build_mst(correlation_matrix):
    """Hitung matriks jarak dari matriks korelasi."""
    distance_matrix = np.sqrt(2 - 2 * correlation_matrix)
    np.fill_diagonal(distance_matrix, 0)
    return distance_matrix


def compute_eigenvector_centrality(distance_matrix):
    """Hitung sentralitas eigenvector dari matriks jarak."""
    adjacency = 1 / (distance_matrix + 1e-8)
    np.fill_diagonal(adjacency, 0)
    eigenvalues, eigenvectors = eigh(adjacency)
    principal_eigenvector     = np.abs(eigenvectors[:, -1])
    centrality                = principal_eigenvector / principal_eigenvector.sum()
    return centrality


def calculate_var(returns, confidence=0.95):
    """Value at Risk pada tingkat kepercayaan tertentu."""
    return np.percentile(returns, (1 - confidence) * 100)


def calculate_rachev_ratio(returns, alpha=0.10):
    """Rachev Ratio = CVaR_upper(alpha) / CVaR_lower(alpha)."""
    threshold_upper = np.percentile(returns, (1 - alpha) * 100)
    threshold_lower = np.percentile(returns, alpha * 100)
    cvar_upper      = returns[returns >= threshold_upper].mean()
    cvar_lower      = abs(returns[returns <= threshold_lower].mean())
    return cvar_upper / cvar_lower if cvar_lower > 0 else 0


def calculate_max_drawdown(cumulative_returns):
    """Maximum drawdown dari seri cumulative returns."""
    running_max = np.maximum.accumulate(cumulative_returns)
    drawdown    = (cumulative_returns - running_max) / running_max
    return drawdown.min()


def get_assets_graph_diversify(returns_window, corr_threshold=0.4):
    """Seleksi aset independen menggunakan Maximum Independent Set (MIS)."""
    corr_mat = returns_window.corr()
    assets   = list(returns_window.mean().sort_values(ascending=False).index)
    G = nx.Graph()
    G.add_nodes_from(assets)
    for i, a1 in enumerate(assets):
        for a2 in assets[i+1:]:
            if abs(corr_mat.loc[a1, a2]) > corr_threshold:
                G.add_edge(a1, a2)
    return list(nx.approximation.maximum_independent_set(G))


def get_assets_graph_diversify_rmt(returns_window, corr_threshold=0.4):
    """Seleksi aset independen menggunakan korelasi yang sudah difilter RMT."""
    assets = returns_window.columns.tolist()
    corr_f = apply_rmt_filter(returns_window)
    G = nx.Graph()
    G.add_nodes_from(assets)
    for i, a1 in enumerate(assets):
        for j, a2 in enumerate(assets):
            if i < j and abs(corr_f[i, j]) > corr_threshold:
                G.add_edge(a1, a2)
    return list(nx.approximation.maximum_independent_set(G))


print("Helper functions defined successfully!")

---
## Sel New — Analisis Dinamika MST (Figure 5)

Analisis *rolling window* untuk menghitung ambang batas MST (*max link distance*) dan koefisien *residuality*, guna memahami evolusi struktur jaringan aset kripto dari waktu ke waktu.

In [ ]:
def calculate_rolling_mst_metrics(returns_df, window=120):
    """Hitung dinamika MST dengan rolling window."""
    dates         = returns_df.index[window:]
    max_links     = []
    residualities = []

    for i in range(window, len(returns_df)):
        window_data  = returns_df.iloc[i - window:i]
        corr_f       = apply_rmt_filter(window_data)
        mst_weights  = build_mst(corr_f)
        max_links.append(np.max(mst_weights))
        residualities.append(np.sum(mst_weights) / (returns_df.shape[1] - 1))

    return pd.DataFrame(
        {'Max Link': max_links, 'Residuality': residualities},
        index=dates
    )


mst_dyn = calculate_rolling_mst_metrics(df_returns)

fig, ax1 = plt.subplots(figsize=(12, 7))
ax1.set_xlabel('Date')
ax1.set_ylabel('Max Link', color='black')
ax1.plot(mst_dyn.index, mst_dyn['Max Link'], color='black', label='Max Link')
ax1.tick_params(axis='y', labelcolor='black')

ax2 = ax1.twinx()
ax2.set_ylabel('Residuality', color='red')
ax2.plot(mst_dyn.index, mst_dyn['Residuality'], color='red', label='Residuality')
ax2.tick_params(axis='y', labelcolor='red')

fig.suptitle('Figure 5 | MST Dynamics: Max Link & Residuality Over Time', fontsize=13)
fig.tight_layout()
plt.show()

---
## Sel 7 — Implementasi Strategi Portofolio

Setiap strategi diatur dalam kelas yang mewarisi `PortfolioStrategy`. Implementasi *Network Markowitz* mencakup penalti sentralitas melalui parameter γ.

In [ ]:
class PortfolioStrategy:
    def __init__(self, name):
        self.name            = name
        self.weights_history = []
        self.returns_history = []

    def get_weights(self, returns_data):
        raise NotImplementedError


# ── 1. Equally Weighted ────────────────────────────────────────────────────────
class EquallyWeighted(PortfolioStrategy):
    def get_weights(self, returns_data):
        n = returns_data.shape[1]
        return np.ones(n) / n


# ── 2. Classical Markowitz ─────────────────────────────────────────────────────
class ClassicalMarkowitz(PortfolioStrategy):
    """Optimasi Mean-Variance tradisional (minimasi varians portofolio)."""
    def get_weights(self, returns_data):
        n  = returns_data.shape[1]
        mu = returns_data.mean().values
        S  = returns_data.cov().values

        objective   = lambda w: w @ S @ w
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n) / n,
                       method='SLSQP', bounds=[(0, 1)] * n,
                       constraints=constraints)
        return res.x if res.success else np.ones(n) / n


# ── 3. Glasso Markowitz ────────────────────────────────────────────────────────
class GlassoMarkowitz(PortfolioStrategy):
    """Markowitz dengan matriks presisi yang diestimasi via Graphical Lasso."""
    def get_weights(self, returns_data):
        n  = returns_data.shape[1]
        mu = returns_data.mean().values
        try:
            glasso = GraphicalLassoCV()
            glasso.fit(returns_data.values)
            S = glasso.covariance_
        except Exception:
            S = returns_data.cov().values

        objective   = lambda w: w @ S @ w
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n) / n,
                       method='SLSQP', bounds=[(0, 1)] * n,
                       constraints=constraints)
        return res.x if res.success else np.ones(n) / n


# ── 4. Network Markowitz ───────────────────────────────────────────────────────
class NetworkMarkowitz(PortfolioStrategy):
    """Network Markowitz: RMT + MST + penalti sentralitas eigenvector."""
    def __init__(self, name="Network Markowitz", gamma=0):
        super().__init__(name)
        self.gamma = gamma

    def get_weights(self, returns_data):
        n   = returns_data.shape[1]
        mu  = returns_data.mean().values
        sig = returns_data.std().values
        Cf  = apply_rmt_filter(returns_data)
        dist = build_mst(Cf)
        cent = compute_eigenvector_centrality(dist)
        Sf   = np.outer(sig, sig) * Cf

        objective   = lambda w: w @ Sf @ w + self.gamma * np.sum(cent * w)
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n) / n,
                       method='SLSQP', bounds=[(0, 1)] * n,
                       constraints=constraints)
        return res.x if res.success else np.ones(n) / n


# ── 5. Graph Diversification ───────────────────────────────────────────────────
class GraphDiversification(PortfolioStrategy):
    """Strategi diversifikasi berbasis graf menggunakan Maximum Independent Set."""
    def __init__(self, name='Graph Diversification', corr_threshold=0.4):
        super().__init__(name)
        self.corr_threshold = corr_threshold

    def get_weights(self, r):
        n      = r.shape[1]
        assets = r.columns.tolist()
        sel    = get_assets_graph_diversify(r, self.corr_threshold)
        w      = np.zeros(n)
        if sel:
            for a in sel:
                w[assets.index(a)] = 1.0 / len(sel)
        else:
            w = np.ones(n) / n
        return w


# ── 6. RMT Graph Diversification ──────────────────────────────────────────────
class RMTGraphDiversification(PortfolioStrategy):
    """Graph Diversification dengan korelasi yang difilter RMT."""
    def __init__(self, name='RMT Graph Diversification', corr_threshold=0.4):
        super().__init__(name)
        self.corr_threshold = corr_threshold

    def get_weights(self, r):
        n      = r.shape[1]
        assets = r.columns.tolist()
        sel    = get_assets_graph_diversify_rmt(r, self.corr_threshold)
        w      = np.zeros(n)
        if sel:
            for a in sel:
                w[assets.index(a)] = 1.0 / len(sel)
        else:
            w = np.ones(n) / n
        return w


print("All portfolio strategy classes defined successfully!")

---
## Sel 8 — Framework Backtesting

Sistem pengujian menggunakan jendela bertahap (*rolling window*) dengan biaya transaksi tetap sebesar 0.1% (10 basis points) untuk merefleksikan kondisi perdagangan riil.

In [ ]:
def backtest_strategy(strategy, df_returns, window_size=120, rebalance_freq=7, transaction_cost=0.001):
    """
    Simulasi backtest dengan rolling window.
    
    Parameters
    ----------
    strategy         : PortfolioStrategy instance
    df_returns       : pd.DataFrame of daily returns
    window_size      : int, panjang window training (default 120 hari)
    rebalance_freq   : int, frekuensi rebalancing dalam hari (default 7)
    transaction_cost : float, biaya transaksi per rebalancing (default 0.001 = 0.1%)
    """
    portfolio_returns = []
    dates             = []

    for i in range(window_size, len(df_returns), rebalance_freq):
        train = df_returns.iloc[i - window_size:i]
        w     = strategy.get_weights(train)

        test_end  = min(i + rebalance_freq, len(df_returns))
        test_data = df_returns.iloc[i:test_end]

        for j in range(len(test_data)):
            daily_ret = np.dot(w, test_data.iloc[j].values)
            if j == 0 and len(portfolio_returns) > 0:
                daily_ret -= transaction_cost
            portfolio_returns.append(daily_ret)
            dates.append(test_data.index[j])

    res_df = pd.DataFrame({'date': dates, 'return': portfolio_returns})
    res_df['cumulative_return'] = (1 + res_df['return']).cumprod()

    return {
        'strategy':          strategy.name,
        'returns':           np.array(portfolio_returns),
        'cumulative_returns': res_df['cumulative_return'].values,
        'results_df':        res_df
    }


print("Backtest framework defined!")

---
## Sel 9 — Eksekusi Backtesting

Simulasi backtesting untuk semua varian strategi, termasuk model usulan dengan berbagai nilai penalti γ.

In [ ]:
# --- Inisialisasi strategi ---
gamma_values = [0.005, 0.025, 0.05, 0.15, 0.7, 1.0]

strategies = (
    [EquallyWeighted("EW"),
     ClassicalMarkowitz("CM"),
     GlassoMarkowitz("GM"),
     NetworkMarkowitz("NW (gamma=0)", gamma=0)]
    + [NetworkMarkowitz(f"NW (gamma={g})", gamma=g) for g in gamma_values]
    + [GraphDiversification('Graph Divers. (theta=0.4)', corr_threshold=0.4),
       GraphDiversification('Graph Divers. (theta=0.5)', corr_threshold=0.5),
       RMTGraphDiversification('RMT GD (theta=0.4)',     corr_threshold=0.4)]
)

# --- Eksekusi backtest ---
results = {}
for strat in strategies:
    print(f"Running: {strat.name} ...", end=' ')
    results[strat.name] = backtest_strategy(strat, df_returns)
    print("Done")

print("\nAll backtests completed!")

---
## Sel 10 — Analisis Performa Periodik (Table 2)

Tabel perbandingan performa kumulatif yang disampel setiap 4 bulan (Januari, Mei, September), mereplikasi format pelaporan pada paper acuan (Tabel 2).

In [ ]:
target_dates = [
    '2018-01-31', '2018-05-31', '2018-09-30',
    '2019-01-31', '2019-05-31', '2019-09-30'
]

table2_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    row    = {'Strategy': strat_name}
    for td in target_dates:
        idx     = df_res.index.searchsorted(pd.Timestamp(td))
        idx     = min(idx, len(df_res) - 1)
        cum_ret = (df_res['cumulative_return'].iloc[idx] - 1) * 100
        row[td] = round(cum_ret, 2)
    table2_rows.append(row)

table2 = pd.DataFrame(table2_rows).set_index('Strategy')
table2.columns = ['Jan-18', 'May-18', 'Sep-18', 'Jan-19', 'May-19', 'Sep-19']

print("TABLE 2 | Cumulative Profits and Losses (%)")
print(table2.to_string())

---
## Sel 11 — Visualisasi Performa Kumulatif (Figure 6)

Evolusi nilai portofolio dengan asumsi investasi awal sebesar 100 USD, untuk periode 7 Januari 2018 – 17 Oktober 2019.

In [ ]:
plt.figure(figsize=(14, 7))
for name, res in results.items():
    # cumulative_returns basis 1.0 × 100 = skala USD
    plt.plot(res['results_df']['date'],
             res['cumulative_returns'] * 100,
             label=name)

plt.title('FIGURE 6 | Performances of Different Portfolio Strategies', fontsize=13)
plt.xlabel('Date')
plt.ylabel('Portfolio Value (USD, initial = 100)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cumulative_returns.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Sel 12 — Analisis Risiko Periodik (Table 3 — VaR)

*Value at Risk* (VaR) pada tingkat kepercayaan 95%, dihitung untuk jendela 4 bulan terakhir di setiap titik sampling. Nilai disajikan dalam nilai absolut × faktor skala 100.

In [ ]:
period_ranges = {
    'Jan-18': ('2017-09-30', '2018-01-31'),
    'May-18': ('2018-01-31', '2018-05-31'),
    'Sep-18': ('2018-05-31', '2018-09-30'),
    'Jan-19': ('2018-09-30', '2019-01-31'),
    'May-19': ('2019-01-31', '2019-05-31'),
    'Sep-19': ('2019-05-31', '2019-09-30'),
}

table3_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    row    = {'Strategy': strat_name}
    for period_label, (start, end) in period_ranges.items():
        segment = df_res.loc[start:end, 'return'].values
        if len(segment) > 0:
            v = abs(calculate_var(segment, 0.95)) * 100
        else:
            v = np.nan
        row[period_label] = round(v, 4)
    table3_rows.append(row)

table3 = pd.DataFrame(table3_rows).set_index('Strategy')

print("TABLE 3 | Value at Risk (95%, 4-month window, scale ×100)")
print(table3.to_string())

---
## Sel 13 — Analisis Risk-Adjusted Return (Table 4 — Sharpe Ratio)

*Sharpe Ratio* dihitung secara periodik untuk mengevaluasi imbal hasil yang disesuaikan dengan risiko.

In [ ]:
table4_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    row    = {'Strategy': strat_name}
    for period_label, (start, end) in period_ranges.items():
        segment = df_res.loc[start:end, 'return'].values
        if len(segment) > 1 and np.std(segment) > 0:
            sr = np.mean(segment) / np.std(segment)
        else:
            sr = np.nan
        row[period_label] = round(sr, 4)
    table4_rows.append(row)

table4 = pd.DataFrame(table4_rows).set_index('Strategy')

print("TABLE 4 | Sharpe Ratio (4-month window)")
print(table4.to_string())

---
## Sel 14 — Analisis Tail-Risk (Table 5 — Rachev Ratio)

*Rachev Ratio* memberikan wawasan tentang asimetri distribusi imbal hasil pada ekor (*tails*), yang umum ditemukan pada pasar kripto.

In [ ]:
table5_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    row    = {'Strategy': strat_name}
    for period_label, (start, end) in period_ranges.items():
        segment = df_res.loc[start:end, 'return'].values
        if len(segment) > 0:
            rr = calculate_rachev_ratio(segment, alpha=0.10)
        else:
            rr = np.nan
        row[period_label] = round(rr, 4)
    table5_rows.append(row)

table5 = pd.DataFrame(table5_rows).set_index('Strategy')

print("TABLE 5 | Rachev Ratio (10%, 4-month window)")
print(table5.to_string())

---
## Sel 15 — Kontribusi Orisinal: Adaptive Graph-Gated Portfolio (AGGP)

Model orisinal tesis yang mengembangkan *Network Markowitz* Giudici melalui mekanisme *gating* adaptif berbasis kepadatan graf temporal (𝒟ₜ). Berbeda dengan model standar yang menggunakan γ statis, AGGP secara otomatis menyesuaikan γ berdasarkan tingkat sinkronisasi pasar.

In [ ]:
class AdaptiveGraphPortfolio(PortfolioStrategy):
    """
    AGGP: Adaptive Graph-Gated Portfolio.
    
    Menggunakan Graph Density sebagai pemicu (trigger) untuk menyesuaikan
    keseimbangan antara optimasi Mean-Variance dan Topologi Graf secara dinamis.
    
    Parameters
    ----------
    sensitivity : float
        Sensitivitas terhadap perubahan struktur korelasi (default 2.0).
        Semakin tinggi → gamma_t semakin besar saat densitas tinggi.
    """
    def __init__(self, name="AGGP (Optimized)", sensitivity=2.0):
        super().__init__(name)
        self.sensitivity = sensitivity

    def get_weights(self, returns_data):
        n  = returns_data.shape[1]
        mu = returns_data.mean().values

        # 1. RMT-Glasso Hybrid Cleaning
        try:
            glasso = GraphicalLassoCV(cv=5)
            glasso.fit(returns_data.values)
            Sf = glasso.covariance_
            Cf = glasso.covariance_ / np.outer(
                np.sqrt(np.diag(glasso.covariance_)),
                np.sqrt(np.diag(glasso.covariance_))
            )
        except Exception:
            Cf = apply_rmt_filter(returns_data)
            Sf = np.outer(returns_data.std(), returns_data.std()) * Cf

        # 2. Dynamic Gating: gamma naik saat pasar tersinkronisasi
        adj     = (np.abs(Cf) > 0.5).astype(int)
        G       = nx.from_numpy_array(adj)
        density = nx.density(G)
        gamma_t = np.clip(density * self.sensitivity, 0.1, 0.9)

        # 3. Network Weights (Centrality Inverse)
        dist  = build_mst(Cf)
        cent  = compute_eigenvector_centrality(dist)
        w_net = 1.0 / (cent + 1e-6)
        w_net /= w_net.sum()

        # 4. Markowitz Weights
        res_mko = minimize(
            lambda w: w @ Sf @ w,
            np.ones(n) / n,
            method='SLSQP',
            bounds=[(0, 1)] * n,
            constraints=({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
        )
        w_mko = res_mko.x if res_mko.success else np.ones(n) / n

        # 5. Adaptive Rebalancing
        return (1 - gamma_t) * w_mko + gamma_t * w_net


print("AGGP class defined!")

### Backtest & Evaluasi AGGP vs Baseline

In [ ]:
aggp = AdaptiveGraphPortfolio("AGGP (sensitivity=2.0)", sensitivity=2.0)

print("Running AGGP backtest...")
results['AGGP'] = backtest_strategy(aggp, df_returns)
print("Done!")

# --- Ringkasan Performa AGGP ---
aggp_returns = results['AGGP']['returns']
aggp_cum     = results['AGGP']['cumulative_returns']

print(f"\n{'='*45}")
print(f"  AGGP Summary")
print(f"{'='*45}")
print(f"  Cumulative Return : {(aggp_cum[-1] - 1)*100:.2f}%")
print(f"  Annualized Return : {np.mean(aggp_returns)*252*100:.2f}%")
print(f"  Annualized Std    : {np.std(aggp_returns)*np.sqrt(252)*100:.2f}%")
print(f"  Sharpe Ratio      : {np.mean(aggp_returns)/np.std(aggp_returns):.4f}")
print(f"  Max Drawdown      : {calculate_max_drawdown(aggp_cum)*100:.2f}%")
print(f"  VaR (95%)         : {abs(calculate_var(aggp_returns, 0.95))*100:.4f}")
print(f"  Rachev Ratio (10%): {calculate_rachev_ratio(aggp_returns, 0.10):.4f}")
print(f"{'='*45}")

In [ ]:
# --- Plot Perbandingan AGGP vs Strategi Utama ---
key_strategies = ['EW', 'CM', 'GM', 'NW (gamma=0)', 'AGGP']

plt.figure(figsize=(13, 6))
for name in key_strategies:
    if name in results:
        res = results[name]
        plt.plot(res['results_df']['date'],
                 res['cumulative_returns'] * 100,
                 label=name, linewidth=1.8)

plt.title('AGGP vs Baseline Strategies | Portfolio Value (initial = 100 USD)', fontsize=12)
plt.xlabel('Date')
plt.ylabel('Portfolio Value (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('aggp_comparison.png', dpi=150, bbox_inches='tight')
plt.show()